In [ ]:
# '취업'에 대해 학습 시키켜록 한다
# 네이버 지식인에서 취업관 관련된 질문들을 가져와서 학습시키기 전 단계까지 처리하려고 한다

# 네이버 지식인에서 취업관 관련된 질문 200개를 추출해서 가져오는 작업
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 매개변수
# 변수명 : 타입 = 기본값
# def 함수() -> 리턴타입:
def get_kin_summary(keyword : str, page : int = 1)->pd.DataFrame:

	# URL 지정 
	url = 'https://kin.naver.com/search/list.naver'

	# url 뒤에 ?변수=값 형태로 만들기 위한 딕셔너리
	# 이 딕셔너리 이름은 반드시 parmas? X
	params = {
		'query' : keyword,
		'page' : page
	}
	# 왼쪽 params는 키워드 인자이기 때문 반드시 params
	response = requests.get(url, params=params)

	try:
		# 상태 코드가 200이면 아무것도 없음. 200이 아니면 예외 발생
		response.raise_for_status()
		# 확인용
		# print(response.text)
		
		# html 코드에서 원하는 요소를 탐색하기 위해 bs를 이용
		soup = BeautifulSoup(response.text, 'lxml') # lxml : html코드 , lxml-xml : 
		
		#._searchListTitleAnchor들을 선택
		# select, select_one : 선택자 기반으로 요소 선택(보통 html에서)
		# find_all, find : 태그 기반으로 요소 선택(보통 xml에서)
		link_elements = soup.select('._searchListTitleAnchor') # 최대 10개

		# 결과를 담을 리스트
		# link_list, title_list = [], []
		data_list = []

		# 반복문으로 선택한 요소를 활용
		for link_element in link_elements:
			# 확인용
			# print(link_element)
			title = link_element.text #요소 안에 있는태그들은 무시한 순수 글자들
			# 요소.get('속성') : 속성 값을 가져옴
			link = link_element.get('href')
			
			# 확인용
			# print(title)
			# print(link)
			
			# 추출한 데이터를 각 리스트에 추가
			data_list.append({
				'title' : title,
				'link' : link
			})

		# 확인용
		# print(link_list)

		# 데이터 처리를 위한 데이터 프레임 생성
		df = pd.DataFrame(data_list) 
		
		# 확인용
		# print(df)
		return df
	except Exception as e:
		print(f'예외 발생 : {e}')
		return pd.DataFrame([])

In [ ]:
def get_kin_full(keyword:str, page:int=1):
	"""지식인 질문들을 질문 제목(title)과 내용(content)으로 구성된 데이터프레임으로 반환하는 함수"""
	df = get_kin_summary(keyword, page)

	list = []
	# df에서 지식인 링크 목록을 가져와서 반복문으로 실행
	for link in df['link']:
		# { 'title' : '질문 전체 제목', 'content' : '질문 내용' }
		dic = get_kin_detail(link)
		list.append(dic)
	
	return pd.DataFrame(list)

def get_kin_detail(url:str):
	response = requests.get(url)

	try:
		response.raise_for_status()

		soup = BeautifulSoup(response.text.strip(), 'lxml')
		
		# .endTitleSection
		title_element = soup.select_one('.endTitleSection')
		
		# 질문 제목 앞에 있는 아이콘 제거(이유 : 아이콘 안에 질문이라는 글자와 고백이 들어감)
		# .iconQuestion
		icon_element = title_element.select_one('.iconQuestion')
		# faq 질문에는 아이콘이 없음 => 아이콘이 있으면 제거
		if icon_element:
			icon_element.decompose()

		title = title_element.text

		# .questionDetail
		content = soup.select_one('.questionDetail').text
		return {
			'title' : title.strip(),
			'content' : content.strip(),
		}

	except Exception as e:	
		print(f'예외 발생{e}')
		return {}


In [ ]:
import time

max_page = 5
keyword = '취업'

# 모든 결과를 담을 데이터프레임
total_df = pd.DataFrame({
	'title' : [], 
	'content' : []
})

# 200개 가져올 때까지 반복 => 기본 10 * 20페이지
for page in range(1, max_page + 1):

	# 키워드, 페이지를 이용하여 검색된 결과에서 제목과 내용을 추출
	df = get_kin_full(keyword, page)

	# 확인용 
	# print(df)
	
	# 각 페이지 결과를 전체에 이어붙임
	total_df = pd.concat([total_df, df], axis=0)
	print(f'{page} 페이지가 로딩되었습니다.')
	# 지연
	time.sleep(0.5)

In [ ]:
# index 리셋
total_df = total_df.reset_index(drop=True)
# print(total_df)

# 추출한 데이터 전처리 작업
# 중복 데이터 제거 : 제목이 중복되면 제거(하나만 남김)
clean_df = total_df.drop_duplicates(subset=['title'])

# 결측치 처리
clean_df = clean_df.dropna()

# 텍스트 길이 분석 후 처리 : AI 학습에서 짧은 문장은 정보량이 부족 => 제거
# 텍스트 길이 평균보다 큰 내용들만 추출

# 텍스트 길이의 평균 
mean_len = clean_df['title'].str.len().mean()

# 평균보다 큰 행들만 추출
final_df = clean_df.loc[clean_df['title'].str.len() > mean_len]

# 인덱스 리셋
final_df = final_df.reset_index(drop=True)
# 현재 : title(질문 일부), link(질문 링크)
# 최종 : title(질문 제목), content(질문 내용)만 확인
# print(final_df)

In [ ]:
# kin_키워드_limit200.csv
final_df.to_csv(f'kin_{keyword}_limit{max_page*10}.csv', index=False, encoding='utf-8-sig')